# Real-Time Fall Detection with YOLOv8
IIT Delhi capstone. Final validation results: **mAP@50 0.909, mAP@50-95 0.614, precision 0.879, recall 0.858, 3.4 ms inference/image on a Tesla T4.**

## 1. Setup and reproducibility

In [ ]:
import os
import glob
import random
import hashlib
import shutil
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import cv2

!pip install ultralytics
!pip install roboflow
import torch
from ultralytics import YOLO
from roboflow import Roboflow

# Same seed -> same data shuffles / augmentation order every run.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Torch    :", torch.__version__)
print("CUDA OK  :", torch.cuda.is_available())
print("Device   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Torch    : 2.10.0+cu128
CUDA OK  : True
Device   : Tesla T4


## 2. Load the dataset
Roboflow Universe *Fall Detection* v1 (CC BY 4.0), exported in YOLOv8 format.

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/Fall Detection.v1-raw-images.yolov8.zip"
EXTRACT_PATH = "/content/Fall_Detection"

os.makedirs(EXTRACT_PATH, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print(f"Error: The zip file specified by ZIP_PATH ('{ZIP_PATH}') does not exist.")
    print("Please ensure the zip file is uploaded to the correct path in your Colab environment.")
else:
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
        print("DATA set extracted successfully")
    except zipfile.BadZipFile:
        print(f"Error: The file '{ZIP_PATH}' is a corrupted or invalid zip file.")
        print("Please check the integrity of the zip file or re-download it.")
    except OSError as e:
        print(f"An OS error occurred during extraction: {e}")
        print("This might indicate issues with the zip file itself, such as corruption, or problems with file permissions or disk space.")

DATA set extracted successfully


In [ ]:
DATA_YML = None

for root, dirs, files in os.walk(EXTRACT_PATH):
  if "data.yaml" in files:
    DATA_YML = os.path.join(root, "data.yaml")
    break

print("data yml found it")
print(DATA_YML)

data yml found it
/content/Fall_Detection/data.yaml


In [ ]:
with open(DATA_YML, "r") as f:
  print(f.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['Fall-Detected']

roboflow:
  workspace: roboflow-universe-projects
  project: fall-detection-ca3o8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/roboflow-universe-projects/fall-detection-ca3o8/dataset/1


## 3. Dataset audit: images, labels and boxes per split

In [ ]:
import glob
from collections import Counter
SPLITS = ["train", "valid", "test"]
DATASET_DIR = EXTRACT_PATH

def count_split(split):
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    imgs    = glob.glob(os.path.join(img_dir, "*.jpg"))
    lbls    = glob.glob(os.path.join(lbl_dir, "*.txt"))
    boxes   = 0
    classes = Counter()
    for lbl in lbls:
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    boxes += 1
                    classes[int(parts[0])] += 1
    return len(imgs), len(lbls), boxes, dict(classes)

print(f"{'split':<8}{'images':>8}{'labels':>8}{'boxes':>8}  classes")
for s in SPLITS:
    n_img, n_lbl, n_box, cls = count_split(s)
    print(f"{s:<8}{n_img:>8}{n_lbl:>8}{n_box:>8}  {cls}")

split     images  labels   boxes  classes
train       3148    3148    3149  {0: 3149}
valid        899     899     899  {0: 899}
test         450     450     450  {0: 450}


## 4. Data-quality pipeline
Remove unreadable images (OpenCV) and exact duplicates (MD5 hash).

In [ ]:
def remove_corrupted_and_duplicates(split):
  img_dir = os.path.join(DATASET_DIR, split, "images")
  lbl_dir = os.path.join(DATASET_DIR, split, "labels")
  seen_hashes = {}
  n_corrupt = n_dup = 0

  for p in sorted(glob.glob(os.path.join(img_dir, "*.jpg"))):
    img = cv2.imread(p)
    if img is None:
      os.remove(p)
      lbl = os.path.join(lbl_dir, os.path.basename(p).replace(".jpg", ".txt"))
      if os.path.exists(lbl): os.remove(lbl)
      n_corrupt += 1
      continue

    with open(p, "rb") as f:
      h = hashlib.md5(f.read()).hexdigest()
      if h in seen_hashes:
        os.remove(p)
        lbl = os.path.join(lbl_dir, os.path.basename(p).replace(".jpg", ".txt"))
        if os.path.exists(lbl): os.remove(lbl)
        n_dup +=1
      else:
        seen_hashes[h] = p

  print(f"[{split}] removed {n_corrupt} corrupted, {n_dup} duplicates")

for s in SPLITS:
    remove_corrupted_and_duplicates(s)

[train] removed 0 corrupted, 0 duplicates
[valid] removed 0 corrupted, 0 duplicates
[test] removed 0 corrupted, 0 duplicates


## 5. Filter outlier boxes by normalised area (train split only)

In [ ]:
def filter_outlier_boxes(split, min_area=0.001, max_area=0.9):
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    kept = removed = 0
    for p in glob.glob(os.path.join(lbl_dir, "*.txt")):
        new_lines = []
        with open(p) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, _, _, w, h = map(float, parts)
                if min_area <= w * h <= max_area:
                    new_lines.append(line)
                    kept += 1
                else:
                    removed += 1
        with open(p, "w") as f:
            f.writelines(new_lines)
    print(f"[{split}] kept {kept} boxes, removed {removed} outlier boxes")

filter_outlier_boxes("train")   # only the training split

[train] kept 3120 boxes, removed 29 outlier boxes


## 6. Visual check of labels

In [ ]:
def draw_yolo_boxes(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w/2) * W); y1 = int((y - h/2) * H)
                x2 = int((x + w/2) * W); y2 = int((y + h/2) * H)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, f"cls {int(cls)}", (x1, max(y1-5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    return img

train_img_dir = os.path.join(DATASET_DIR, "train", "images")
train_lbl_dir = os.path.join(DATASET_DIR, "train", "labels")
sample_imgs   = random.sample(glob.glob(os.path.join(train_img_dir, "*.jpg")), 4)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, p in zip(axes.flatten(), sample_imgs):
    lbl = os.path.join(train_lbl_dir, os.path.basename(p).replace(".jpg", ".txt"))
    ax.imshow(draw_yolo_boxes(p, lbl)); ax.axis('off')
plt.tight_layout(); plt.show()


## 7. Train YOLOv8s (100 epochs, AdamW, cosine LR, augmentation)

In [ ]:
# ============================================
# YOLOv8 Fall Detection Training (Crash Safe)
# ============================================

# Install ultralytics (run once)
!pip install ultralytics -q

# ============================================
# Imports
# ============================================

from google.colab import drive  # optional: save runs to Drive
from ultralytics import YOLO
import os

# ============================================
# Mount Google Drive
# ============================================

drive.mount('/content/drive')

# ============================================
# Dataset YAML Path
# Example:
# DATA_YML = "/content/drive/MyDrive/fall_dataset/data.yaml"
# ============================================

DATA_YML = os.path.join(EXTRACT_PATH, "data.yaml")

# ============================================
# Create Model
# yolov8s.pt = small pretrained model
# ============================================

model = YOLO("yolov8s.pt")

# ============================================
# Train Model
# ============================================

results = model.train(

    # Dataset config
    data=DATA_YML,

    # Training settings
    epochs=100,
    imgsz=640,
    batch=16,

    # GPU
    device='0',

    # Early stopping
    patience=20,

    # ========================================
    # Optimizer
    # ========================================

    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    weight_decay=0.0005,

    # ========================================
    # Augmentations
    # Helps reduce overfitting
    # ========================================

    mosaic=1.0,
    mixup=0.1,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # ========================================
    # Save directly to Google Drive
    # ========================================

    project="/content/drive/MyDrive/fall_runs",

    name="exp_v8s_improved",

    # Save checkpoint every epoch
    save_period=1,

    # Generate training plots
    plots=True,

    # Reproducibility
    seed=42,

    # Save best and last models
    save=True
)

# ============================================
# Print Saved Location
# ============================================

print("\nTraining Finished!")
print("Model Folder:")
print(results.save_dir)

# ============================================
# Best Model Path
# ============================================

best_model = f"{results.save_dir}/weights/best.pt"

print("\nBest Model Saved At:")
print(best_model)

# ============================================
# Last Model Path
# ============================================

last_model = f"{results.save_dir}/weights/last.pt"

print("\nLast Checkpoint Saved At:")
print(last_model)

... (log truncated) ...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     26/100      5.44G      1.299      1.213      1.559         31        640: 100% ━━━━━━━━━━━━ 197/197 2.9it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.3it/s 6.8s
                   all        899        899      0.785      0.795      0.827      0.501

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     27/100      5.44G      1.297      1.208      1.558         32        640: 100% ━━━━━━━━━━━━ 197/197 2.9it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.2it/s 7.0s
                   all        899        899      0.809      0.817      0.858      0.506

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     28/100      5.44G      1.276      1.175      1.543         30     

## 8. Inference on video and images

In [ ]:
# Run the trained model on your own video (path is an example)
results = model.predict(
    source="/content/sample_video.mp4",
    conf=0.5,
    save=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/fall_runs/exp_v8s_improved/weights/best.pt  # or weights/best.pt from this repo's release")

results = model.predict(
    source="/content/Fall_Detection/test/images/people-1021-_jpg.rf.c750cd4de10b8e1471793aeffead3ca9.jpg",
    conf=0.5,
    save=True
)


image 1/1 /content/Fall_Detection/test/images/people-1021-_jpg.rf.c750cd4de10b8e1471793aeffead3ca9.jpg: 640x480 1 Fall-Detected, 72.2ms
Speed: 3.3ms preprocess, 72.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict-2
